# Student Performance Prediction
## Notebook 4 — Model Evaluation & Feature Importance

**Author:** Abdifatah Muhlar
**Data Source:** UCI Machine Learning Repository — Student Performance Dataset

---

### Objective
This notebook performs deep evaluation of the selected Random Forest
model. We analyze feature importance to understand which factors most
strongly predict student performance, examine the model's decision
boundaries, and translate technical findings into actionable
educational recommendations.

### Why Random Forest Was Selected
- Highest cross-validation score (71.2%)
- Highest recall (86.8%) — correctly identifies most at-risk students
- Lowest variance across folds (std=0.038) — most consistent model
- Handles mixed feature types effectively

In [1]:
# ============================================================
# SECTION 1 — Import Libraries & Rebuild Model
# ============================================================

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score,
                             classification_report)
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Load data
X = pd.read_csv('student_features.csv')
y = pd.read_csv('student_target.csv').squeeze()

# Rebuild train/test split with same random state
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Rebuild Random Forest model
rf_model = RandomForestClassifier(random_state=42, n_estimators=100)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

print("Random Forest model rebuilt successfully")
print(f"Training set : {X_train.shape[0]} students")
print(f"Test set     : {X_test.shape[0]} students")
print(f"\nModel Performance:")
print(f"  Accuracy  : {accuracy_score(y_test, y_pred):.3f}")
print(f"  Precision : {precision_score(y_test, y_pred):.3f}")
print(f"  Recall    : {recall_score(y_test, y_pred):.3f}")
print(f"  F1 Score  : {f1_score(y_test, y_pred):.3f}")

Random Forest model rebuilt successfully
Training set : 316 students
Test set     : 79 students

Model Performance:
  Accuracy  : 0.671
  Precision : 0.708
  Recall    : 0.868
  F1 Score  : 0.780


---
## Section 2 — Feature Importance Analysis

Feature importance measures how much each variable contributes
to the Random Forest model's predictions. Higher importance means
the feature is more useful for distinguishing passing from failing students.

This is one of the most valuable outputs of the entire project —
it tells us not just whether we can predict performance, but
exactly which factors drive it.

In [2]:
# ============================================================
# SECTION 2 — Feature Importance Analysis
# ============================================================

# Get feature importances
importances = rf_model.feature_importances_
feature_names = X.columns.tolist()

# Create sorted dataframe
feat_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=True)

# Top 15 features
top15 = feat_df.tail(15)

# Plot
fig = go.Figure()

fig.add_trace(go.Bar(
    x=top15['Importance'],
    y=top15['Feature'],
    orientation='h',
    marker=dict(
        color=top15['Importance'],
        colorscale='Blues',
        showscale=False
    ),
    text=[f'{v:.3f}' for v in top15['Importance']],
    textposition='outside',
    hovertemplate='%{y}<br>Importance: %{x:.4f}<extra></extra>'
))

fig.update_layout(
    title=dict(
        text='Top 15 Most Important Features — Random Forest',
        font=dict(size=16), x=0.5, xanchor='center'
    ),
    xaxis=dict(
        title='Feature Importance Score',
        showgrid=True,
        gridcolor='lightgrey'
    ),
    yaxis=dict(showgrid=False, tickfont=dict(size=12)),
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=550,
    margin=dict(l=120, r=100, t=80, b=60)
)

fig.show()

# Print top 15
print("TOP 15 MOST IMPORTANT FEATURES")
print("=" * 40)
for _, row in feat_df.tail(15).iloc[::-1].iterrows():
    print(f"{row['Feature']:<15} : {row['Importance']:.4f}")

TOP 15 MOST IMPORTANT FEATURES
absences        : 0.1085
failures        : 0.1051
goout           : 0.0557
age             : 0.0542
Mjob            : 0.0457
health          : 0.0454
Medu            : 0.0412
freetime        : 0.0403
Fedu            : 0.0389
Fjob            : 0.0373
Walc            : 0.0372
famrel          : 0.0368
studytime       : 0.0348
reason          : 0.0317
guardian        : 0.0275


### Insight — Feature Importance Analysis

Top 15 Features Ranked by Importance:
1.  absences  (0.1085) — strongest predictor
2.  failures  (0.1051) — second strongest
3.  goout     (0.0557) — social behavior signal
4.  age       (0.0542) — older students at higher risk
5.  Mjob      (0.0457) — mother's occupation
6.  health    (0.0454) — student health status
7.  Medu      (0.0412) — mother's education level
8.  freetime  (0.0403) — time management signal
9.  Fedu      (0.0389) — father's education level
10. Fjob      (0.0373) — father's occupation
11. Walc      (0.0372) — weekend alcohol consumption
12. famrel    (0.0368) — family relationship quality
13. studytime (0.0348) — weekly study hours
14. reason    (0.0317) — reason for choosing school
15. guardian  (0.0275) — student's guardian

Educational Interpretation:

1. Absences — The Single Strongest Predictor (0.1085)
   School attendance is the most powerful signal in the entire
   model. Every missed class represents lost instruction time
   and compounds knowledge gaps. More importantly, absences are
   an observable, real-time indicator — schools can monitor this
   daily and trigger interventions immediately when patterns emerge.
   This finding supports mandatory attendance tracking systems
   with automated early warning alerts.

2. Past Failures — Compounding Disadvantage (0.1051)
   Prior academic failure is almost equally predictive as absences.
   Students who have failed before carry knowledge gaps, reduced
   confidence, and lower motivation into subsequent years. This
   finding argues strongly for preventing first failures through
   early support rather than remediation after the fact. The cost
   of intervention before failure is far lower than the cost of
   recovery after it.

3. Going Out Frequency (0.0557)
   Social behavior outside school is the third most important
   predictor — more important than study time. This counterintuitive
   finding suggests that how students spend their time outside
   school is as critical as what happens inside it. Schools and
   parents need to work together on structured after-school
   environments that balance social development with academic focus.

4. Parental Factors (Mjob, Medu, Fedu, Fjob)
   Four of the top 15 features relate to parental background.
   This reflects the home learning environment — students with
   educated, professionally employed parents receive more academic
   support, encouragement, and exposure to achievement culture.
   This equity dimension highlights the need for schools to
   actively compensate for home environment disadvantages through
   targeted support programs.

5. Study Time Ranked 13th
   Surprisingly, study time is only the 13th most important feature.
   This does not mean studying is unimportant — it means that
   behavioral and demographic context predicts outcomes more
   strongly than self-reported study hours alone. Quality and
   consistency of study matters more than quantity.

Conclusion:
The two most actionable intervention points are absences and
past failures — both directly observable by schools in real time.
A simple early warning system tracking these two variables alone
could identify the majority of at-risk students before final exams.

---
## Section 3 — Learning Curve Analysis

A learning curve shows how model performance changes as we
increase the amount of training data. It helps us understand:
- Whether the model would benefit from more data
- Whether the model is overfitting or underfitting
- The stability of model performance

In [3]:
# ============================================================
# SECTION 3 — Learning Curve Analysis
# ============================================================

train_sizes, train_scores, val_scores = learning_curve(
    rf_model, X, y,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5,
    scoring='accuracy',
    random_state=42
)

train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)

fig = go.Figure()

# Training score
fig.add_trace(go.Scatter(
    x=train_sizes,
    y=train_mean,
    mode='lines+markers',
    name='Training Score',
    line=dict(color='royalblue', width=2),
    marker=dict(size=8),
    hovertemplate='Training size: %{x}<br>Score: %{y:.3f}<extra></extra>'
))

# Training confidence band
fig.add_trace(go.Scatter(
    x=list(train_sizes) + list(train_sizes[::-1]),
    y=list(train_mean + train_std) + list(train_mean - train_std)[::-1],
    fill='toself',
    fillcolor='rgba(65, 105, 225, 0.15)',
    line=dict(color='rgba(255,255,255,0)'),
    name='Training ±1 std',
    hoverinfo='skip'
))

# Validation score
fig.add_trace(go.Scatter(
    x=train_sizes,
    y=val_mean,
    mode='lines+markers',
    name='Validation Score',
    line=dict(color='crimson', width=2),
    marker=dict(size=8),
    hovertemplate='Training size: %{x}<br>Score: %{y:.3f}<extra></extra>'
))

# Validation confidence band
fig.add_trace(go.Scatter(
    x=list(train_sizes) + list(train_sizes[::-1]),
    y=list(val_mean + val_std) + list(val_mean - val_std)[::-1],
    fill='toself',
    fillcolor='rgba(220, 20, 60, 0.15)',
    line=dict(color='rgba(255,255,255,0)'),
    name='Validation ±1 std',
    hoverinfo='skip'
))

fig.update_layout(
    title=dict(
        text='Learning Curve — Random Forest Model',
        font=dict(size=16), x=0.5, xanchor='center'
    ),
    xaxis=dict(
        title='Training Set Size',
        showgrid=True,
        gridcolor='lightgrey'
    ),
    yaxis=dict(
        title='Accuracy Score',
        showgrid=True,
        gridcolor='lightgrey',
        range=[0.5, 1.05]
    ),
    hovermode='x unified',
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(x=0.65, y=0.25),
    height=500
)

fig.show()
print("Learning curve rendered")

Learning curve rendered


### Insight — Learning Curve Analysis

Observed Pattern:
- Training score starts high (~1.0) with small data and decreases
- Validation score starts low and increases as more data is added
- The two curves converge as training size approaches 316 students

Interpretation:

1. Convergence — Healthy Model Behavior
   The gap between training and validation scores narrows
   consistently as training size increases. This is the hallmark
   of a well-behaved model that is learning genuine patterns
   rather than memorizing training data.

2. Mild Overfitting
   A gap remains between training and validation scores even
   at full training size. This is expected for Random Forest —
   ensemble models naturally fit training data well. The gap
   is not large enough to be concerning.

3. Would More Data Help?
   The validation score is still slightly increasing at maximum
   training size, suggesting the model could benefit from
   additional student records. With 500-600 students the model
   would likely reach 73-75% validation accuracy. This is a
   practical recommendation for schools implementing this system —
   collect more historical data to improve prediction reliability.

4. Stability
   The narrow confidence bands (shaded regions) indicate
   consistent performance across all 5 cross-validation folds —
   confirming the model is stable and not sensitive to which
   students end up in the training set.

---
## Section 4 — Real World Prediction Demonstration

We demonstrate the model's practical application by creating
fictional student profiles and predicting their pass/fail outcome.
This simulates how the model would be used in a real school
intervention system.

In [4]:
# ============================================================
# SECTION 4 — Real World Prediction Demonstration
# ============================================================

# Create 5 fictional student profiles
# Using same feature order as training data
feature_names = X.columns.tolist()

# Profile definitions
profiles = {
    'Student A — High Risk': {
        'school': 0, 'sex': 1, 'age': 17, 'address': 0,
        'famsize': 0, 'Pstatus': 0, 'Medu': 1, 'Fedu': 1,
        'Mjob': 0, 'Fjob': 0, 'reason': 2, 'guardian': 1,
        'traveltime': 3, 'studytime': 1, 'failures': 2,
        'schoolsup': 0, 'famsup': 0, 'paid': 0,
        'activities': 0, 'nursery': 0, 'higher': 0,
        'internet': 0, 'romantic': 1, 'famrel': 2,
        'freetime': 5, 'goout': 5, 'Dalc': 4, 'Walc': 5,
        'health': 2, 'absences': 20
    },
    'Student B — Medium Risk': {
        'school': 0, 'sex': 0, 'age': 16, 'address': 1,
        'famsize': 0, 'Pstatus': 1, 'Medu': 2, 'Fedu': 2,
        'Mjob': 2, 'Fjob': 2, 'reason': 0, 'guardian': 1,
        'traveltime': 2, 'studytime': 2, 'failures': 1,
        'schoolsup': 1, 'famsup': 1, 'paid': 0,
        'activities': 1, 'nursery': 1, 'higher': 1,
        'internet': 1, 'romantic': 0, 'famrel': 3,
        'freetime': 3, 'goout': 3, 'Dalc': 2, 'Walc': 2,
        'health': 3, 'absences': 8
    },
    'Student C — Low Risk': {
        'school': 0, 'sex': 0, 'age': 16, 'address': 1,
        'famsize': 1, 'Pstatus': 1, 'Medu': 4, 'Fedu': 4,
        'Mjob': 4, 'Fjob': 3, 'reason': 3, 'guardian': 1,
        'traveltime': 1, 'studytime': 4, 'failures': 0,
        'schoolsup': 0, 'famsup': 1, 'paid': 1,
        'activities': 1, 'nursery': 1, 'higher': 1,
        'internet': 1, 'romantic': 0, 'famrel': 5,
        'freetime': 2, 'goout': 1, 'Dalc': 1, 'Walc': 1,
        'health': 5, 'absences': 1
    },
    'Student D — Borderline': {
        'school': 1, 'sex': 1, 'age': 17, 'address': 1,
        'famsize': 0, 'Pstatus': 1, 'Medu': 2, 'Fedu': 2,
        'Mjob': 2, 'Fjob': 2, 'reason': 1, 'guardian': 0,
        'traveltime': 2, 'studytime': 2, 'failures': 1,
        'schoolsup': 1, 'famsup': 0, 'paid': 1,
        'activities': 0, 'nursery': 1, 'higher': 1,
        'internet': 1, 'romantic': 1, 'famrel': 3,
        'freetime': 3, 'goout': 3, 'Dalc': 2, 'Walc': 3,
        'health': 3, 'absences': 6
    },
    'Student E — Strong Performer': {
        'school': 0, 'sex': 0, 'age': 15, 'address': 1,
        'famsize': 1, 'Pstatus': 1, 'Medu': 4, 'Fedu': 3,
        'Mjob': 1, 'Fjob': 4, 'reason': 3, 'guardian': 1,
        'traveltime': 1, 'studytime': 4, 'failures': 0,
        'schoolsup': 0, 'famsup': 1, 'paid': 0,
        'activities': 1, 'nursery': 1, 'higher': 1,
        'internet': 1, 'romantic': 0, 'famrel': 5,
        'freetime': 2, 'goout': 1, 'Dalc': 1, 'Walc': 1,
        'health': 5, 'absences': 0
    }
}

# Create dataframe and predict
profiles_df = pd.DataFrame(profiles).T[feature_names]
predictions = rf_model.predict(profiles_df)
probabilities = rf_model.predict_proba(profiles_df)

print("REAL WORLD PREDICTION RESULTS")
print("=" * 60)
print(f"{'Student':<30} {'Prediction':<12} {'Pass Prob':<12} {'Fail Prob'}")
print("-" * 60)
for i, (name, pred) in enumerate(zip(profiles.keys(), predictions)):
    pass_prob = probabilities[i][1]
    fail_prob = probabilities[i][0]
    result = "PASS ✓" if pred == 1 else "FAIL ✗"
    print(f"{name:<30} {result:<12} {pass_prob:.1%}{'':>6} {fail_prob:.1%}")

REAL WORLD PREDICTION RESULTS
Student                        Prediction   Pass Prob    Fail Prob
------------------------------------------------------------
Student A — High Risk          FAIL ✗       39.0%       61.0%
Student B — Medium Risk        PASS ✓       51.0%       49.0%
Student C — Low Risk           PASS ✓       82.0%       18.0%
Student D — Borderline         PASS ✓       63.0%       37.0%
Student E — Strong Performer   PASS ✓       79.0%       21.0%


In [5]:
# ============================================================
# VISUALIZATION — Prediction Probability Chart
# ============================================================

student_names = [s.split('—')[0].strip() for s in profiles.keys()]
pass_probs = [probabilities[i][1] for i in range(len(profiles))]
fail_probs = [probabilities[i][0] for i in range(len(profiles))]
pred_colors = ['crimson' if p == 0 else 'steelblue' for p in predictions]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=student_names,
    y=pass_probs,
    name='Pass Probability',
    marker_color=pred_colors,
    text=[f'{p:.0%}' for p in pass_probs],
    textposition='outside',
    hovertemplate='%{x}<br>Pass Probability: %{y:.1%}<extra></extra>'
))

fig.add_hline(
    y=0.5,
    line_dash='dash',
    line_color='black',
    line_width=1.5,
    annotation_text='Decision Boundary (50%)',
    annotation_position='top right'
)

fig.update_layout(
    title=dict(
        text='Predicted Pass Probability — Student Profiles',
        font=dict(size=16), x=0.5, xanchor='center'
    ),
    xaxis=dict(title='Student Profile', showgrid=False),
    yaxis=dict(
        title='Pass Probability',
        showgrid=True,
        gridcolor='lightgrey',
        range=[0, 1.15],
        tickformat='.0%'
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    showlegend=False,
    height=500
)

fig.show()
print("Prediction chart rendered")

Prediction chart rendered


### Insight — Real World Prediction Demonstration

Prediction Results:
- Student A (High Risk)        : FAIL — 39% pass probability
- Student B (Medium Risk)      : PASS — 51% pass probability
- Student C (Low Risk)         : PASS — 82% pass probability
- Student D (Borderline)       : PASS — 63% pass probability
- Student E (Strong Performer) : PASS — 79% pass probability

Interpretation:

1. Student A — High Risk (FAIL, 39%)
   High absences (20), 2 past failures, high alcohol consumption,
   frequent social outings, no internet access, rural address,
   and low parental education combine to create the highest risk
   profile. This student needs immediate intervention — counseling,
   attendance monitoring, and academic support before exams.

2. Student B — Medium Risk (PASS, 51%)
   Only 51% pass probability — effectively a coin flip. One past
   failure and moderate social behavior place this student at the
   edge of the decision boundary. Small improvements in attendance
   and study time could significantly shift this outcome. This
   student represents the highest-value intervention target —
   modest support could prevent a failure.

3. Student C — Low Risk (PASS, 82%)
   Zero failures, low absences, high study time, highly educated
   parents, and minimal alcohol consumption produce a strong
   performance profile. This student requires no intervention.

4. Student D — Borderline (PASS, 63%)
   One past failure and moderate risk behaviors create uncertainty.
   The model predicts a pass but with meaningful doubt. Monitoring
   attendance and providing optional academic support would be
   prudent for this student.

5. Student E — Strong Performer (PASS, 79%)
   Zero failures, zero absences, maximum study time, and strong
   family support produce a confident pass prediction. This
   student profile represents the target outcome for school
   intervention programs.

Practical Application:
This demonstration shows how the model can be deployed as a
real-time student monitoring tool. By inputting observable
student data — attendance, past performance, family background —
schools can generate pass probability scores for every student
at the start of each term, enabling targeted support allocation
before academic outcomes are determined.

In [6]:
# ============================================================
# SECTION 5 — FINAL PROJECT SUMMARY
# ============================================================

print("""
STUDENT PERFORMANCE PREDICTION — FINAL SUMMARY
================================================

Project         : Binary Classification — Pass/Fail Prediction
Dataset         : UCI Student Performance (Mathematics)
Students        : 395 | Features : 30 | Target : Pass/Fail

PIPELINE:
  1. Data Cleaning     — encoding, feature engineering, leakage prevention
  2. EDA               — grade distribution, correlations, behavioral analysis
  3. Modeling          — Logistic Regression, Decision Tree, Random Forest
  4. Evaluation        — feature importance, learning curves, live prediction

FINAL MODEL: Random Forest (n_estimators=100)

Performance:
  Accuracy          : 67.1%
  Precision         : 70.8%
  Recall            : 86.8%
  F1 Score          : 78.0%
  CV Mean           : 71.2%
  CV Std Dev        : 0.038

Top 5 Predictive Features:
  1. absences  (0.1085) — strongest signal
  2. failures  (0.1051) — compounding disadvantage
  3. goout     (0.0557) — social behavior
  4. age       (0.0542) — maturity factor
  5. Mjob      (0.0457) — family background

Key Findings:
  1. Absences and past failures are the two most powerful
     predictors of student academic outcomes
  2. Social behavior (going out, alcohol) matters more
     than study time in predicting performance
  3. Parental background accounts for 4 of the top 15
     features — highlighting structural equity issues
  4. The model achieves 86.8% recall — correctly identifying
     the vast majority of at-risk students
  5. Random Forest outperforms all other models on
     generalization (CV 71.2%) and recall (86.8%)

Policy Recommendations:
  1. Implement real-time attendance monitoring with
     automated alerts for at-risk patterns
  2. Prioritize intervention after first failure —
     prevent compounding disadvantage
  3. Develop structured after-school programs to
     reduce unproductive social time
  4. Create targeted support for students from
     lower parental education backgrounds
  5. Deploy this model as a term-start screening
     tool to allocate support resources efficiently

Practical Application:
  A school deploying this model at the start of each term
  could identify 87% of students at risk of failing —
  enabling proactive support allocation before academic
  outcomes are determined.
""")


STUDENT PERFORMANCE PREDICTION — FINAL SUMMARY

Project         : Binary Classification — Pass/Fail Prediction
Dataset         : UCI Student Performance (Mathematics)
Students        : 395 | Features : 30 | Target : Pass/Fail

PIPELINE:
  1. Data Cleaning     — encoding, feature engineering, leakage prevention
  2. EDA               — grade distribution, correlations, behavioral analysis
  3. Modeling          — Logistic Regression, Decision Tree, Random Forest
  4. Evaluation        — feature importance, learning curves, live prediction

FINAL MODEL: Random Forest (n_estimators=100)

Performance:
  Accuracy          : 67.1%
  Precision         : 70.8%
  Recall            : 86.8%
  F1 Score          : 78.0%
  CV Mean           : 71.2%
  CV Std Dev        : 0.038

Top 5 Predictive Features:
  1. absences  (0.1085) — strongest signal
  2. failures  (0.1051) — compounding disadvantage
  3. goout     (0.0557) — social behavior
  4. age       (0.0542) — maturity factor
  5. Mjob      (0.04